In [ ]:
#plot_weekly_dry_day_distribution.py

import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings #import warnings module to manage warnings

#--- Configuration ---
#this notebook lives in 03b/dry_day_analysis/ (moved 2026-08-17); the output
#bundle stays at the stage-folder root, hence the leading "../".
INPUT_CSV_FILENAME = "../created_dfs_weekly_dry_day_analysis/df_weekly_dry_days_paper.csv"
OUTPUT_DIR = "../created_dfs_weekly_dry_day_analysis/weekly_dry_day_plots" #directory to save plots

#state FIPS mapping (adjust if the state set changes)
STATE_MAP = {
    '17': 'Illinois',
    '18': 'Indiana',
    '19': 'Iowa',
    '27': 'Minnesota',
    '31': 'Nebraska'
}
STATES_TO_PLOT = ['17', '18', '19', '27', '31'] #FIPS prefixes for states

#weekly dry-day column patterns. Accepts BOTH namings, so this notebook works
#against either vintage of the CSV:
#CDD_1mm_W1              - produced after the rename fix (2026-08-17)
#cDD_1mm_Wweek_in_gs_1.0 - produced by the earlier buggy rename
#the counts were identical either way; only the column labels differed.
PATTERN_1MM = r"CDD_1mm_W(?:week_in_gs_)?(\d+)(?:\.\d+)?$"
PATTERN_2MM = r"CDD_2mm_W(?:week_in_gs_)?(\d+)(?:\.\d+)?$"

#--- Helper Functions ---

def get_base_path():
    """Determines the script's directory for robust path handling."""
    try:
        #standard execution
        return os.path.dirname(__file__)
    except NameError:
        #interactive execution (like Jupyter, etc.)
        return os.getcwd()

def plot_distribution(data, title, filename, output_dir):
    """Generates and saves a bar plot of average dry days per week."""
    if data.empty:
        print(f"Skipping plot '{title}' - No data available.")
        return

    #data 'week' column is already int from melt step, ensure sorting
    data_sorted = data.sort_values('week')
    week_order = sorted(data_sorted['week'].unique()) #get order for seaborn plot

    plt.figure(figsize=(15, 6))

    #--- UPDATED SEABORN CALL ---
    #use 'hue' and 'legend=False' as suggested by FutureWarning
    #also pass the sorted week order to ensure correct plotting
    sns.barplot(x='week', y='avg_dry_days', data=data_sorted, hue='week',
                palette='Blues_d', order=week_order, legend=False, dodge=False)
    #--- END UPDATE ---

    plt.title(title, fontsize=16)
    plt.xlabel("Week within Growing Season (April 1st = Start of Week 1)", fontsize=12)
    plt.ylabel("Average Number of Dry Days per Week", fontsize=12)
    #Adjust xticks if there are many weeks
    if len(week_order) > 20:
         plt.xticks(ticks=range(len(week_order)), labels=week_order, rotation=90, ha='center', fontsize=9) #use ticks and labels for clarity
    else:
         plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(fontsize=10)
    plt.ylim(0, 7) #max 7 dry days in a week
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()

    #ensure output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath)
    print(f"Saved plot: {filepath}")
    plt.close() #close the plot to free memory

#--- Main Execution ---

def main():
    #--- Suppress specific warnings ---
    #Optional: Suppress the SettingWithCopyWarning if you understand the context
    pd.options.mode.chained_assignment = None #be cautious using this globally
    #Optional: Suppress the specific FutureWarning from seaborn
    warnings.simplefilter(action='ignore', category=FutureWarning)
    #--- End Warning Suppression ---

    print("--- Starting Weekly Dry Day Distribution Plotting ---")

    script_dir = get_base_path()
    input_path = os.path.join(script_dir, INPUT_CSV_FILENAME)
    plot_output_dir = os.path.join(script_dir, OUTPUT_DIR)

    if not os.path.exists(input_path):
        print(f"Error: Input file not found at: {input_path}")
        return

    print(f"Loading data from: {input_path}")
    try:
        df = pd.read_csv(input_path)
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return

    #--- Data Preparation ---
    if 'fips_full' not in df.columns:
        print("Error: 'fips_full' column missing.")
        return

    #extract state FIPS
    df['state_fips'] = df['fips_full'].astype(str).str[:2]

    pattern_1mm = PATTERN_1MM
    pattern_2mm = PATTERN_2MM

    #identify weekly columns
    weekly_cols_1mm = sorted(
        [col for col in df.columns if re.match(pattern_1mm, col)],
        key=lambda x: int(re.search(pattern_1mm, x).group(1))
    )
    weekly_cols_2mm = sorted(
        [col for col in df.columns if re.match(pattern_2mm, col)],
        key=lambda x: int(re.search(pattern_2mm, x).group(1))
    )

    if not weekly_cols_1mm or not weekly_cols_2mm:
        print(f"Error: Could not find weekly dry day columns matching 'CDD_?mm_W<week>' or 'CDD_?mm_Wweek_in_gs_##.0'. Found columns: {list(df.columns)}")
        return

    print(f"Found {len(weekly_cols_1mm)} weekly columns (1mm), {len(weekly_cols_2mm)} (2mm).")

    #melt data for 1mm threshold
    df_melted_1mm = df.melt(
        id_vars=['fips_full', 'year', 'state_fips'],
        value_vars=weekly_cols_1mm,
        var_name='week_col',
        value_name='dry_days_count'
    )
    df_melted_1mm['week'] = df_melted_1mm['week_col'].str.extract(pattern_1mm).astype(int)
    df_melted_1mm['threshold'] = '1mm'

    #Melt data for 2mm threshold
    df_melted_2mm = df.melt(
        id_vars=['fips_full', 'year', 'state_fips'],
        value_vars=weekly_cols_2mm,
        var_name='week_col',
        value_name='dry_days_count'
    )
    df_melted_2mm['week'] = df_melted_2mm['week_col'].str.extract(pattern_2mm).astype(int)
    df_melted_2mm['threshold'] = '2mm'

    #combine melted data
    df_melted = pd.concat([df_melted_1mm, df_melted_2mm], ignore_index=True)
    df_melted = df_melted.drop(columns=['week_col'])
    df_melted = df_melted.dropna(subset=['dry_days_count'])

    print("Data melted successfully.")

    #--- Plotting ---

    #plot Overall Distribution
    print("\nCalculating and plotting overall distribution...")
    df_overall_avg = df_melted.groupby(['threshold', 'week'])['dry_days_count'].mean().reset_index()
    df_overall_avg.rename(columns={'dry_days_count': 'avg_dry_days'}, inplace=True)

    plot_distribution(
        df_overall_avg[df_overall_avg['threshold'] == '1mm'],
        f"Average Weekly Dry Days (<1mm) - All {len(STATES_TO_PLOT)} States",
        "overall_dist_1mm.png",
        plot_output_dir
    )
    plot_distribution(
        df_overall_avg[df_overall_avg['threshold'] == '2mm'],
        f"Average Weekly Dry Days (<2mm) - All {len(STATES_TO_PLOT)} States",
        "overall_dist_2mm.png",
        plot_output_dir
    )

    #plot Distribution per State
    print("\nCalculating and plotting distribution per state...")
    for state_fips in STATES_TO_PLOT:
        state_name = STATE_MAP.get(state_fips, f"State_{state_fips}")
        print(f" Processing {state_name}...")

        df_state = df_melted[df_melted['state_fips'] == state_fips]

        if df_state.empty:
            print(f"  No data found for state {state_name}. Skipping.")
            continue

        df_state_avg = df_state.groupby(['threshold', 'week'])['dry_days_count'].mean().reset_index()
        df_state_avg.rename(columns={'dry_days_count': 'avg_dry_days'}, inplace=True)

        plot_distribution(
            df_state_avg[df_state_avg['threshold'] == '1mm'],
            f"Average Weekly Dry Days (<1mm) - {state_name}",
            f"{state_name.lower()}_dist_1mm.png",
            plot_output_dir
        )
        plot_distribution(
            df_state_avg[df_state_avg['threshold'] == '2mm'],
            f"Average Weekly Dry Days (<2mm) - {state_name}",
            f"{state_name.lower()}_dist_2mm.png",
            plot_output_dir
        )

    print("\n--- Plotting complete. ---")

if __name__ == "__main__":
    main()

In [ ]:
#plot_yearly_total_distribution.py

import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings #import warnings module to manage warnings

#--- Configuration ---
#this notebook lives in 03b/dry_day_analysis/ (moved 2026-08-17); the output
#bundle stays at the stage-folder root, hence the leading "../".
INPUT_CSV_FILENAME = "../created_dfs_weekly_dry_day_analysis/df_weekly_dry_days_paper.csv"
OUTPUT_DIR = "../created_dfs_weekly_dry_day_analysis/yearly_total_dry_day_plots" #directory to save plots

#state FIPS mapping (adjust if the state set changes)
STATE_MAP = {
    '17': 'Illinois',
    '18': 'Indiana',
    '19': 'Iowa',
    '27': 'Minnesota',
    '31': 'Nebraska'
}
STATES_TO_PLOT = ['17', '18', '19', '27', '31'] #FIPS prefixes for states

#weekly dry-day column patterns. Accepts BOTH namings, so this notebook works
#against either vintage of the CSV:
#CDD_1mm_W1              - produced after the rename fix (2026-08-17)
#cDD_1mm_Wweek_in_gs_1.0 - produced by the earlier buggy rename
#the counts were identical either way; only the column labels differed.
PATTERN_1MM = r"CDD_1mm_W(?:week_in_gs_)?(\d+)(?:\.\d+)?$"
PATTERN_2MM = r"CDD_2mm_W(?:week_in_gs_)?(\d+)(?:\.\d+)?$"

#--- Helper Functions ---

def get_base_path():
    """Determines the script's directory for robust path handling."""
    try:
        #standard execution
        return os.path.dirname(__file__)
    except NameError:
        #interactive execution (like Jupyter, etc.)
        return os.getcwd()

def plot_yearly_total_cdd(data, title, filename, output_dir):
    """Generates and saves a line plot of average total GS dry days per year."""
    if data.empty:
        print(f"Skipping plot '{title}' - No data available.")
        return

    plt.figure(figsize=(15, 6))

    #use seaborn lineplot
    sns.lineplot(x='year', y='avg_total_dry_days', data=data, marker='o')

    plt.title(title, fontsize=16)
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Average Total Dry Days in Growing Season", fontsize=12)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(axis='both', linestyle='--', alpha=0.7)
    plt.tight_layout()

    #ensure output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath)
    print(f"Saved plot: {filepath}")
    plt.close()

#--- Main Execution ---

def main():
    print("--- Starting Yearly Total Dry Day Distribution Plotting ---")

    script_dir = get_base_path()
    input_path = os.path.join(script_dir, INPUT_CSV_FILENAME)
    plot_output_dir = os.path.join(script_dir, OUTPUT_DIR)

    if not os.path.exists(input_path):
        print(f"Error: Input file not found at: {input_path}")
        return

    print(f"Loading data from: {input_path}")
    try:
        df = pd.read_csv(input_path)
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return

    #--- Data Preparation ---
    if 'fips_full' not in df.columns:
        print("Error: 'fips_full' column missing.")
        return

    df['state_fips'] = df['fips_full'].astype(str).str[:2]

    pattern_1mm = PATTERN_1MM
    pattern_2mm = PATTERN_2MM

    weekly_cols_1mm = [col for col in df.columns if re.match(pattern_1mm, col)]
    weekly_cols_2mm = [col for col in df.columns if re.match(pattern_2mm, col)]

    if not weekly_cols_1mm or not weekly_cols_2mm:
        print(f"Error: Could not find weekly dry day columns matching 'CDD_?mm_W<week>' or 'CDD_?mm_Wweek_in_gs_##.0'. Found columns: {list(df.columns)}")
        return

    print(f"Found {len(weekly_cols_1mm)} weekly columns (1mm), {len(weekly_cols_2mm)} (2mm).")

    #--- Calculate Yearly Totals ---
    print("\nCalculating yearly total dry days...")
    df['total_cdd_1mm'] = df[weekly_cols_1mm].sum(axis=1)
    df['total_cdd_2mm'] = df[weekly_cols_2mm].sum(axis=1)

    #--- Average Yearly Totals per State ---
    print("Calculating state averages per year...")
    df_yearly_state_avg = df.groupby(['state_fips', 'year'])[['total_cdd_1mm', 'total_cdd_2mm']].mean().reset_index()
    df_yearly_state_avg.rename(columns={'total_cdd_1mm': 'avg_total_dry_days_1mm',
                                          'total_cdd_2mm': 'avg_total_dry_days_2mm'}, inplace=True)

    #--- Plotting Yearly Total Distributions ---
    print("\nPlotting average yearly total dry days per state...")
    for state_fips in STATES_TO_PLOT:
        state_name = STATE_MAP.get(state_fips, f"State_{state_fips}")
        print(f" Processing {state_name}...")

        df_yearly_state = df_yearly_state_avg[df_yearly_state_avg['state_fips'] == state_fips]

        if df_yearly_state.empty:
             print(f"  No yearly data found for state {state_name}. Skipping.")
             continue

        #plot for 1mm threshold
        plot_yearly_total_cdd(
            df_yearly_state[['year', 'avg_total_dry_days_1mm']].rename(columns={'avg_total_dry_days_1mm': 'avg_total_dry_days'}),
            f"Average Total GS Dry Days (<1mm) per Year - {state_name}",
            f"{state_name.lower()}_yearly_total_1mm.png",
            plot_output_dir
        )
        #plot for 2mm threshold
        plot_yearly_total_cdd(
            df_yearly_state[['year', 'avg_total_dry_days_2mm']].rename(columns={'avg_total_dry_days_2mm': 'avg_total_dry_days'}),
            f"Average Total GS Dry Days (<2mm) per Year - {state_name}",
            f"{state_name.lower()}_yearly_total_2mm.png",
            plot_output_dir
        )

    print("\n--- Plotting complete. ---")

if __name__ == "__main__":
    main()